In [1]:
from pathlib import Path

In [2]:
region_list = ["Groningen en NO-Drenthe","Noord-Westelijke Delta","Overijsselse Vecht","Limburg","Vallei en Veluwe","Achterhoek", "Brabantse Delta","Friesland","Limburg","ARK-NZK",  "Noord-Brabant Oost",
               "Rivierenland","Scheldestromen"
               ]

In [3]:
for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Aggregated_schakels.gpkg"

Processing region: Groningen en NO-Drenthe network
Processing region: Noord-Westelijke Delta network
Processing region: Overijsselse Vecht network
Processing region: Limburg network
Processing region: Vallei en Veluwe network
Processing region: Achterhoek network
Processing region: Brabantse Delta network
Processing region: Friesland network
Processing region: Limburg network
Processing region: ARK-NZK network
Processing region: Noord-Brabant Oost network
Processing region: Rivierenland network
Processing region: Scheldestromen network


In [11]:


# New cell: merge all regional Aggregated_schakels.gpkg into one (Amersfoort / RD New, EPSG:28992)
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd

TARGET_CRS = "EPSG:28992"  # Amersfoort / RD New

keep_cols = [
    "NETWERKSCH",
    "total_length",
    "flooded_length",
    "bridge_length_sum",
    "tunnel_length_sum",
    "total_damage",
    "F_EV2_ma_max",
    "geometry",
]
additive_cols = ["flooded_length", "bridge_length_sum", "tunnel_length_sum", "total_damage"]
numeric_cols = ["total_length"] + additive_cols + ["F_EV2_ma_max"]

gdfs = []

for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    gpkg_path = root_dir / "Aggregated_schakels.gpkg"
    if not gpkg_path.exists():
        warnings.warn(f"Missing file, skipping: {gpkg_path}")
        continue

    try:
        gdf = gpd.read_file(gpkg_path)  # assumes a single layer
    except Exception as e:
        warnings.warn(f"Failed to read {gpkg_path}: {e}")
        continue

    if "NETWERKSCH" not in gdf.columns:
        warnings.warn(f"'NETWERKSCH' column missing in {gpkg_path}, skipping")
        continue
    if gdf.crs is None:
        #warnings.warn(f"CRS missing in {gpkg_path}, skipping (cannot ensure Amersfoort projection).")
        #continue
        warnings.warn(f"CRS missing in {gpkg_path}, assuming Amersfoort / RD New ({TARGET_CRS}).")
        # Assign CRS metadata without reprojection (assumes data are already in Amersfoort)
        gdf.set_crs(TARGET_CRS, inplace=True)

    # Reproject to Amersfoort / RD New
    if gdf.crs.to_string() != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)

    # Ensure required columns exist with defaults
    for col in additive_cols:
        if col not in gdf.columns:
            gdf[col] = 0.0
    if "total_length" not in gdf.columns:
        gdf["total_length"] = np.nan
    if "F_EV2_ma_max" not in gdf.columns:
        gdf["F_EV2_ma_max"] = np.nan

    # Numeric dtypes
    for col in numeric_cols:
        gdf[col] = pd.to_numeric(gdf[col], errors="coerce")

    # Keep only needed columns (create missing ones)
    for col in keep_cols:
        if col not in gdf.columns:
            gdf[col] = 0.0 if col in additive_cols else (np.nan if col != "geometry" else gdf.geometry)

    gdfs.append(gdf[keep_cols])

if not gdfs:
    raise SystemExit("No GeoPackages found to merge.")

df = pd.concat(gdfs, ignore_index=True)

# Aggregate by NETWERKSCH
g = df.groupby("NETWERKSCH", dropna=False)

def pick_total_length(s: pd.Series) -> float:
    vals = pd.to_numeric(s, errors="coerce").dropna().to_numpy()
    if len(vals) == 0:
        return np.nan
    v0 = float(vals[0])
    if not np.allclose(vals, v0, rtol=1e-6, atol=1e-6):
        warnings.warn(f"Inconsistent total_length for NETWERKSCH='{s.name}': {vals.tolist()}")
    return v0

# Geometry union
geom_df = df[["NETWERKSCH", "geometry"]].dissolve(by="NETWERKSCH")

# Summations
sums_df = g[additive_cols].sum(min_count=1)

# Max of F_EV2_ma_max
max_df = g["F_EV2_ma_max"].max().rename("F_EV2_ma_max")

# Total length consistency
totlen_df = g["total_length"].apply(pick_total_length).to_frame(name="total_length")

# Combine
merged = geom_df.join([totlen_df, sums_df, max_df])
merged = merged.reset_index()
merged = merged[keep_cols]

# Ensure GeoDataFrame with Amersfoort CRS
merged = gpd.GeoDataFrame(merged, geometry="geometry", crs=TARGET_CRS)
merged['fraction_flooded'] = (merged['flooded_length'] / merged['total_length']) 
merged['dam_m'] = merged['total_damage'] / merged['flooded_length']

# New: derive NET (without -L/-R/-M) and type (the removed suffix)
_tmp = merged["NETWERKSCH"].astype("string")
merged["type"] = _tmp.str.extract(r"-(L|R|M)$")[0]          # L, R or M (or NaN if none)
merged["NET"]  = _tmp.str.replace(r"-(L|R|M)$", "", regex=True)
del _tmp

# Write output
out_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "Aggregated_schakels_merged.gpkg"
merged.to_file(out_path, driver="GPKG", layer="Aggregated_schakels_merged")

print(f"Merged {len(df)} rows into {len(merged)} NETWERKSCH groups.")
print(f"Wrote (CRS={TARGET_CRS}): {out_path}")

print(f"Merged {len(df)} rows into {len(merged)} NETWERKSCH groups.")
print(f"Wrote (CRS={TARGET_CRS}): {out_path}")

# New: NET-level aggregation for damages
geom_by_net = merged[["NET", "geometry"]].dissolve(by="NET")
sums_by_net = (
    merged.groupby("NET")[["total_damage", "flooded_length"]]
          .sum(min_count=1)
)
damages = geom_by_net.join(sums_by_net).reset_index()
damages = gpd.GeoDataFrame(damages, geometry="geometry", crs=TARGET_CRS)

# Recalculate dam_m at NET level
damages["dam_m"] = damages["total_damage"] / damages["flooded_length"]
damages.loc[damages["flooded_length"].isna() | (damages["flooded_length"] == 0), "dam_m"] = np.nan

# Write damages file
damages_out = out_dir / "damages_Aggregated_to_schakel.gpkg"
damages.to_file(damages_out, driver="GPKG", layer="damages")
print(f"Wrote NET-level damages to: {damages_out}")


Processing region: Groningen en NO-Drenthe network
Processing region: Noord-Westelijke Delta network


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_33372\677907050.py:45: UserWarning: CRS missing in P:\bovenregionale-stresstest-hwn\Analysis\Noord-Westelijke Delta\Outputs\Aggregated_schakels.gpkg, assuming Amersfoort / RD New (EPSG:28992).
  warnings.warn(f"CRS missing in {gpkg_path}, assuming Amersfoort / RD New ({TARGET_CRS}).")


Processing region: Overijsselse Vecht network
Processing region: Limburg network
Processing region: Vallei en Veluwe network
Processing region: Achterhoek network


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_33372\677907050.py:45: UserWarning: CRS missing in P:\bovenregionale-stresstest-hwn\Analysis\Achterhoek\Outputs\Aggregated_schakels.gpkg, assuming Amersfoort / RD New (EPSG:28992).
  warnings.warn(f"CRS missing in {gpkg_path}, assuming Amersfoort / RD New ({TARGET_CRS}).")


Processing region: Brabantse Delta network
Processing region: Friesland network
Processing region: Limburg network
Processing region: ARK-NZK network
Processing region: Noord-Brabant Oost network
Processing region: Rivierenland network
Processing region: Scheldestromen network


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged')) failed: unable to open database file"


Merged 609 rows into 478 NETWERKSCH groups.
Wrote (CRS=EPSG:28992): P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged.gpkg
Merged 609 rows into 478 NETWERKSCH groups.
Wrote (CRS=EPSG:28992): P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged.gpkg
Wrote NET-level damages to: P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\damages_Aggregated_to_schakel.gpkg


In [10]:
print(merged.columns)

Index(['NETWERKSCH', 'total_length', 'flooded_length', 'bridge_length_sum',
       'tunnel_length_sum', 'total_damage', 'F_EV2_ma_max', 'geometry',
       'fraction_flooded', 'dam_m', 'Areas', 'Areas_name'],
      dtype='object')


In [12]:


def add_area_overlay(gdf: gpd.GeoDataFrame, area_path: Path, target_crs: str, key_col: str) -> gpd.GeoDataFrame:
    import fiona

    if not area_path.exists():
        warnings.warn(f"Area file not found: {area_path}. Skipping area overlay.")
        return gdf.copy()

    layers = fiona.listlayers(str(area_path))
    layer = layers[0]
    area = gpd.read_file(area_path, layer=layer)

    # Ensure CRS
    if area.crs is None:
        warnings.warn(f"CRS missing in {area_path}, assuming {target_crs}.")
        area.set_crs(target_crs, inplace=True)
    if area.crs.to_string() != target_crs:
        area = area.to_crs(target_crs)

    # Pick a human-readable label column if present
    candidate_labels = ["name", "id"]
    label_col = next((c for c in candidate_labels if c in area.columns), None)

    new_col = area_path.stem  # e.g., "Areas"

    # Spatial join (intersects)
    left = gdf[[key_col, "geometry"]].copy()
    right_cols = ["geometry"] + ([label_col] if label_col else [])
    sj = gpd.sjoin(left, area[right_cols], how="left", predicate="intersects")

    # Boolean overlap indicator per key
    has_overlap = (
        sj.groupby(key_col)["index_right"]
          .apply(lambda s: s.notna().any())
          .rename(new_col)
          .reset_index()
    )

    out = gdf.merge(has_overlap, on=key_col, how="left")
    out[new_col] = out[new_col].fillna(False)

    # Optional: concatenate labels of overlapping polygons
    if label_col:
        labels = (
            sj.groupby(key_col)[label_col]
              .apply(lambda s: ";".join(sorted({str(v) for v in s.dropna()})) if s.notna().any() else None)
              .rename(f"{new_col}_{label_col}")
              .reset_index()
        )
        out = out.merge(labels, on=key_col, how="left")

    return out


def write_with_area(gdf: gpd.GeoDataFrame, out_dir: Path, base_name: str, area_path: Path, target_crs: str, key_col: str):
    out_dir.mkdir(parents=True, exist_ok=True)

    # Enrich with area overlay
    enriched = add_area_overlay(gdf, area_path, target_crs, key_col)

    # GPKG
    gpkg_path = out_dir / f"{base_name}.gpkg"
    layer_name = base_name.replace(" ", "_")
    enriched.to_file(gpkg_path, driver="GPKG", layer=layer_name)

    # SHP (cast bools)
    shp_ready = enriched.copy()
    for col in shp_ready.select_dtypes(include=["bool"]).columns:
        shp_ready[col] = shp_ready[col].astype("uint8")
    shp_path = out_dir / f"{base_name}.shp"
    shp_ready.to_file(shp_path, driver="ESRI Shapefile")

    print(f"Wrote: {gpkg_path} (layer={layer_name}) and {shp_path}")
    return enriched, gpkg_path, shp_path

# --- Use the function for both merged and damages ---
area_gpkg = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")

# For merged (key: NETWERKSCH)
_merged_enriched, _, _ = write_with_area(
    merged, out_dir, "Aggregated_schakels_merged_with_area", area_gpkg, TARGET_CRS, key_col="NETWERKSCH"
)

# For damages (key: NET)
_damages_enriched, _, _ = write_with_area(
    damages, out_dir, "damages_with_area", area_gpkg, TARGET_CRS, key_col="NET"
)


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Aggregated_schakels_merged_with_area')) failed: unable to open database file"
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_33372\1366941704.py:70: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  shp_ready.to_file(shp_path, driver="ESRI Shapefile")


Wrote: P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area.gpkg (layer=Aggregated_schakels_merged_with_area) and P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Aggregated_schakels_merged_with_area.shp


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_33372\1366941704.py:70: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  shp_ready.to_file(shp_path, driver="ESRI Shapefile")


Wrote: P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\damages_with_area.gpkg (layer=damages_with_area) and P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\damages_with_area.shp


In [8]:

from pathlib import Path as _Path
import fiona

area_gpkg = _Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Areas.gpkg")  # <- adjust path
if area_gpkg.exists():
    layers = fiona.listlayers(str(area_gpkg))
    layer = layers[0]
    area = gpd.read_file(area_gpkg, layer=layer)

    # Force Amersfoort / RD New
    if area.crs is None:
        warnings.warn(f"CRS missing in {area_gpkg}, assuming {TARGET_CRS}.")
        area.set_crs(TARGET_CRS, inplace=True)
    if area.crs.to_string() != TARGET_CRS:
        area = area.to_crs(TARGET_CRS)

    # Pick a human-readable label column if present
    candidate_labels = ["name", "id"]
    label_col = next((c for c in candidate_labels if c in area.columns), None)

    new_col = area_gpkg.stem  # column named from file, e.g., "area"

    # Spatial join (intersects). Keep one row per NETWERKSCH by aggregating.
    left = merged[["NETWERKSCH", "geometry"]].copy()
    right_cols = ["geometry"] + ([label_col] if label_col else [])
    sj = gpd.sjoin(left, area[right_cols], how="left", predicate="intersects")

    # Boolean overlap indicator
    has_overlap = sj.groupby("NETWERKSCH")["index_right"].apply(lambda s: s.notna().any()).rename(new_col)
    merged = merged.merge(has_overlap.reset_index(), on="NETWERKSCH", how="left")
    merged[new_col] = merged[new_col].fillna(False)

    # Optional: concatenate labels of overlapping polygons
    if label_col:
        labels = (
            sj.groupby("NETWERKSCH")[label_col]
              .apply(lambda s: ";".join(sorted({str(v) for v in s.dropna()})) if s.notna().any() else None)
              .rename(f"{new_col}_{label_col}")
              .reset_index()
        )
        merged = merged.merge(labels, on="NETWERKSCH", how="left")
else:
    warnings.warn(f"Area file not found: {area_gpkg}. Skipping area overlay.")

# Write output
out_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "Aggregated_schakels_merged_with area.gpkg"
merged.to_file(out_path, driver="GPKG")

for col in merged.select_dtypes(include=["bool"]).columns:
    merged[col] = merged[col].astype("uint8")

out_path = out_dir / "Aggregated_schakels_merged_with_area.shp"
merged.to_file(out_path, driver="ESRI Shapefile")

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 478 WHERE lower(table_name) = lower('Aggregated_schakels_merged_with area')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 478 WHERE lower(table_name) = lower('Aggregated_schakels_merged_with area')) failed: unable to open database file"
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_33372\3924818031.py:55: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  merged.to_file(out_path, driver="ESRI Shapefile")
